### Load Data:

In [138]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from nltk.stem import LancasterStemmer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string
from sklearn.model_selection import train_test_split

In [139]:
# Load raw data
amazon_data = pd.read_csv('train_amazon.csv')
amazon_test = pd.read_csv('test_amazon.csv')
movie_data = pd.read_csv("train.csv")

In [140]:
amazon_data.loc[len(amazon_data)] = amazon_data.columns
amazon_data.columns = ["polarity","title",'review']
amazon_data["polarity"] = amazon_data["polarity"].astype(int)
amazon_data["polarity"] = amazon_data["polarity"].replace({1:"Negative",2:"Positive"})
amazon_data["datatype"] = "Train"
amazon_data["reviewtype"] = "Product"
amazon_data = amazon_data.drop("title", axis=1)
amazon_data = pd.concat([amazon_data[amazon_data["polarity"]=="Positive"].head(100000),amazon_data[amazon_data["polarity"]=="Negative"].head(100000)]).reset_index(drop=True)
amazon_data

,polarity,review,datatype,reviewtype
0,Positive,I'm reading a lot of reviews saying that this ...,Train,Product
1,Positive,This soundtrack is my favorite music of all ti...,Train,Product
2,Positive,I truly like this soundtrack and I enjoy video...,Train,Product
3,Positive,"If you've played the game, you know how divine...",Train,Product
4,Positive,I am quite sure any of you actually taking the...,Train,Product
...,...,...,...,...
199995,Negative,"I had this album for a while, but I eventually...",Train,Product
199996,Negative,This album has absolutely no original songs in...,Train,Product
199997,Negative,This is Kiss at their worst. I heard them on t...,Train,Product
199998,Negative,This has always been an odd album to me. It's ...,Train,Product


In [141]:
amazon_test.loc[len(amazon_test)] = amazon_test.columns
amazon_test.columns = ["polarity","title",'review']
amazon_test["polarity"] = amazon_test["polarity"].astype(int)
amazon_test["polarity"] = amazon_test["polarity"].replace({1:"Negative",2:"Positive"})
amazon_test["datatype"] = "Test"
amazon_test["reviewtype"] = "Product"
amazon_test = amazon_test.drop("title", axis=1)
amazon_test = pd.concat([amazon_test[amazon_test["polarity"]=="Positive"].head(20000),amazon_test[amazon_test["polarity"]=="Negative"].head(20000)]).reset_index(drop=True)
amazon_test

,polarity,review,datatype,reviewtype
0,Positive,Despite the fact that I have only played a sma...,Test,Product
1,Positive,Check out Maha Energy's website. Their Powerex...,Test,Product
2,Positive,Reviewed quite a bit of the combo players and ...,Test,Product
3,Positive,"Exotic tales of the Orient from the 1930's. ""D...",Test,Product
4,Positive,"I currently live in Europe, and this is the bo...",Test,Product
...,...,...,...,...
39995,Negative,This is a great story and I was happy to see c...,Test,Product
39996,Negative,"Reading this was a chore, but I did it just to...",Test,Product
39997,Negative,Unbelievable. I own hard copies of every book ...,Test,Product
39998,Negative,If this manuscript was sent to Little Brown by...,Test,Product


In [142]:
movie_data = movie_data.drop("Unnamed: 0", axis=1)
movie_data.columns = ["review","polarity"]
movie_data["polarity"] = movie_data["polarity"].astype(str)
movie_data["reviewtype"] = "Movies"

# split them equally so the case balance is preserved in split
tta, ta = train_test_split(movie_data, test_size=0.15, stratify=movie_data['polarity'], random_state=42)
tta['datatype'] = 'Train'
ta['datatype'] = 'Test'

movie_data_stratified = pd.concat([tta, ta]).reset_index(drop=True)
movie_data_stratified = movie_data_stratified[["polarity","review","datatype","reviewtype"]]
movie_data_stratified

,polarity,review,datatype,reviewtype
0,neg,No idea how this is rated as high as it is (5....,Train,Movies
1,neg,I just blew four dollars renting this movie! W...,Train,Movies
2,neg,"As others have mentioned, this movie is simila...",Train,Movies
3,pos,Begotten is black and white distorted images. ...,Train,Movies
4,pos,First of all - I'm not one to go all sappy ove...,Train,Movies
...,...,...,...,...
39995,pos,Paul Verhoeven's De Vierde Man (The Fourth Man...,Test,Movies
39996,neg,"This movie could have been an impressing epic,...",Test,Movies
39997,pos,10/10 for this film.<br /><br />i'm a british ...,Test,Movies
39998,neg,"First off, the title character is not even the...",Test,Movies


In [143]:
data = pd.concat([amazon_data,amazon_test,movie_data_stratified]).reset_index(drop=True)
data

,polarity,review,datatype,reviewtype
0,Positive,I'm reading a lot of reviews saying that this ...,Train,Product
1,Positive,This soundtrack is my favorite music of all ti...,Train,Product
2,Positive,I truly like this soundtrack and I enjoy video...,Train,Product
3,Positive,"If you've played the game, you know how divine...",Train,Product
4,Positive,I am quite sure any of you actually taking the...,Train,Product
...,...,...,...,...
279995,pos,Paul Verhoeven's De Vierde Man (The Fourth Man...,Test,Movies
279996,neg,"This movie could have been an impressing epic,...",Test,Movies
279997,pos,10/10 for this film.<br /><br />i'm a british ...,Test,Movies
279998,neg,"First off, the title character is not even the...",Test,Movies


In [144]:
# data.groupby(["reviewtype","polarity","datatype"]).count()

### Split Data

In [145]:
train_text = data[data['datatype']=="Train"]["review"].tolist()
train_labels = data[data['datatype']=="Train"]['polarity'].tolist()

test_text = data[data['datatype']=="Test"]["review"].tolist()
test_labels = data[data['datatype']=="Test"]['polarity'].tolist()

### Text cleaning and preprocessing:

In [146]:
# Initialize the lemmatizer and stopword list
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text_nltk(text_list):
    '''Function to preprocess a list of texts by cleaning, lemmatizing, and removing unnecessary elements using NLTK.'''
    processed_texts = []

    for text in text_list:
        text = text.lower()
        tokens = word_tokenize(text)
        tokens = [
            lemmatizer.lemmatize(token) for token in tokens
            if token not in stop_words               # Remove stopwords
            and token not in string.punctuation      # Remove punctuation
            and token.isalpha()                      # Keep only alphabetic words (no digits or symbols)
        ]
        
        processed_texts.append(" ".join(tokens))

    return processed_texts

# Preprocess the training and test data
train_text_processed = preprocess_text_nltk(train_text)
test_text_processed = preprocess_text_nltk(test_text)

### Vectorizer

In [147]:
# TF-IDF Vectorization
max_feature_num = 500
vectorizer = TfidfVectorizer(max_features=max_feature_num)

In [148]:
# Fit and transform training data, and transform test data
train_vec = vectorizer.fit_transform(train_text_processed)
test_vec = vectorizer.transform(test_text_processed)

In [149]:
# Check the shape of train_vec to confirm it's 2D
print("Shape of train_vec:", train_vec.shape)
print("Shape of test_vec:", test_vec.shape)

Shape of train_vec: (234000, 500)
Shape of test_vec: (46000, 500)


### Model Training

In [150]:
# Model Training 
clf = MultinomialNB().fit(train_vec, train_labels)

In [151]:
# Predict on training data
train_pred = clf.predict(train_vec)

# Predict on test data
test_pred = clf.predict(test_vec)

In [152]:
# Accuracy
train_accuracy = accuracy_score(train_labels, train_pred)
test_accuracy = accuracy_score(test_labels, test_pred)

In [153]:
# Classification Report (Precision, Recall, F1-Score)
train_classification_report = classification_report(train_labels, train_pred)
test_classification_report = classification_report(test_labels, test_pred)

In [154]:
# Confusion Matrix
train_confusion_matrix = confusion_matrix(train_labels, train_pred)
test_confusion_matrix = confusion_matrix(test_labels, test_pred)

In [156]:
# ROC-AUC score for binary classification (POS/NEG)
train_roc_auc = roc_auc_score(train_labels, clf.predict_proba(train_vec)[:, 1])
test_roc_auc = roc_auc_score(test_labels, clf.predict_proba(test_vec)[:, 1])

ValueError: multi_class must be in ('ovo', 'ovr')

In [157]:
# Print the Results
print("Training Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

print("\nTraining Classification Report:")
print(train_classification_report)

print("\nTest Classification Report:")
print(test_classification_report)

Training Accuracy: 0.7417820512820513
Test Accuracy: 0.7432391304347826

Training Classification Report:
              precision    recall  f1-score   support

    Negative       0.74      0.78      0.76    100000
    Positive       0.74      0.78      0.76    100000
         neg       0.79      0.52      0.62     17000
         pos       0.77      0.51      0.61     17000

    accuracy                           0.74    234000
   macro avg       0.76      0.65      0.69    234000
weighted avg       0.74      0.74      0.74    234000


Test Classification Report:
              precision    recall  f1-score   support

    Negative       0.74      0.78      0.76     20000
    Positive       0.74      0.78      0.76     20000
         neg       0.80      0.52      0.63      3000
         pos       0.78      0.51      0.61      3000

    accuracy                           0.74     46000
   macro avg       0.76      0.65      0.69     46000
weighted avg       0.75      0.74      0.74     460

In [158]:
# def plot_confusion_matrix(conf_matrix, title="Confusion Matrix"):
#     sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
#     plt.title(title)
#     plt.ylabel('Actual')
#     plt.xlabel('Predicted')
#     plt.show()
# 
# # Plot confusion matrices
# plot_confusion_matrix(train_confusion_matrix, title="Training Set Confusion Matrix")
# plot_confusion_matrix(test_confusion_matrix, title="Test Set Confusion Matrix")


In [159]:
# Plot ROC Curve
def plot_roc_curve(fpr, tpr, roc_auc, title="Receiver Operating Characteristic (ROC) Curve"):
    plt.figure(figsize=(10, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.show()

In [160]:
# # Compute ROC curve and AUC for Train and Test
# fpr_train, tpr_train, _ = roc_curve(train_labels, clf.predict_proba(train_vec)[:, 1], pos_label='pos')
# fpr_test, tpr_test, _ = roc_curve(test_labels, clf.predict_proba(test_vec)[:, 1], pos_label='pos')


In [161]:
# # Plot ROC for Train and Test
# plot_roc_curve(fpr_train, tpr_train, train_roc_auc, title="Train ROC Curve")
# plot_roc_curve(fpr_test, tpr_test, test_roc_auc, title="Test ROC Curve")


In [162]:
# Insights on overfitting/underfitting
if train_accuracy > test_accuracy:
    print("\nModel may be overfitting: Training accuracy is  higher than test accuracy.")
elif train_accuracy < test_accuracy:
    print("\nModel may be underfitting: Test accuracy is higher than training accuracy.")
else:
    print("\nModel is performing consistently across both train and test sets.")


Model may be underfitting: Test accuracy is higher than training accuracy.


In [163]:
# Further insights into overfitting or underfitting
if train_accuracy > test_accuracy and test_accuracy < 0.7:
    print("\nThis could indicate overfitting. You may need to try regularization or reduce model complexity.")
elif train_accuracy < 0.7 and test_accuracy < 0.7:
    print("\nThis could indicate underfitting. You may need to use a more complex model or adjust the feature extraction.")
else:
    print("\nModel seems well-balanced.")


Model seems well-balanced.


### 1.6. Creating a Re-Usable Model called Pickle
##### Pickle is kind of a dictionary that stores the trained model and can be used again instead of training it over and over.

In [164]:
import pickle

# save model and other necessary modules
all_info_want_to_save = {
    'model': clf,
    'vectorizer': vectorizer
}
save_path = open("sample_trained_model.pickle","wb")
pickle.dump(all_info_want_to_save, save_path)


In [165]:
pickle.load(open('sample_trained_model.pickle', "rb"))['vectorizer']

TfidfVectorizer(max_features=500)

### Demonstrate

In [166]:
# Function to load the model and vectorizer, and make predictions
def predict_polarity(model_path, user_query):
    saved_model_dic = pickle.load(open(model_path, "rb"))
    saved_clf = saved_model_dic['model']
    saved_vectorizer = saved_model_dic['vectorizer']
    
    # Preprocess the user query
    preprocessed_query = preprocess_text_nltk([user_query])
    
    # Transform the query text using the saved vectorizer
    query_vec = saved_vectorizer.transform(preprocessed_query)
    
    # Predict the polarity
    prediction = saved_clf.predict(query_vec)
    
    return prediction[0]

In [172]:
# Main function to interact with the user
def main():
    # User input query
    user_query = input("Enter your query: ")
    
    # Load the model and make prediction
    polarity = predict_polarity("sample_trained_model.pickle", user_query)
    
    # Output the polarity
    print(f"Polarity of your query: {polarity}")

if __name__ == "__main__":
    main()

Polarity of your query: Positive
